# HAM10000 ResNet Fine-tuning
This notebook demonstrates how to fine-tune a ResNet model on the HAM10000 dataset using the `ResNetFineTuneTrainer` class.

In [1]:
import sys
from pathlib import Path

current_dir = Path.cwd()
proj_dir = current_dir.parent.parent.resolve() # get project root directory

print(f"Project directory set to: {proj_dir}")

# Add the project root to the system path to import local modules
sys.path.append(str(proj_dir / 'python'))

Project directory set to: C:\Users\T22127A\Documents\priv\Uni\priv_dev\dermascan


In [2]:
from dataset_manager import DatasetManager
from pipeline.resnet_finetune_train import ResNetFineTuneTrainer

## Prepare data
Initialize > Load > Split in train, val, test

In [3]:
# Initialize the dataset manager
data_dir = str(proj_dir / "data")
manager = DatasetManager(data_dir=data_dir)

# Load metadata (filtering for Melanoma vs Nevi as per trainer restrictions)
manager.load_cleaned_metadata(filter_melanoma_only=True)

# Split the data into train (70%), validation (15%), and test (15%) sets
train_df, val_df, test_df = manager.train_test_val_split(val_size=0.15, test_size=0.15)

print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Test samples: {len(test_df)}")

Training samples: 5470
Validation samples: 1173
Test samples: 1173


## Confugure ResNetFineTuneTrainer

In [ ]:
trainer = ResNetFineTuneTrainer()

# Set up the model (ResNet50, 2 classes: nv vs mel)
trainer.set_model(model_name='resnet50', pretrained=True) # , input_channels=6

# Set data loaders
trainer.set_data_loaders(train_df, val_df, test_df, filter_name='lbp_filter', data_path=data_dir)

# Configure optimizer and scheduler
trainer.configure_optimizer(optimizer_type='adam', lr=1e-4, weight_decay=1e-4)
trainer.configure_scheduler(scheduler_type='reduce_on_plateau', patience=3)

# Set output paths for artifacts
trainer.set_output_paths(
    model_save_dir=str(proj_dir / "python" / "results" / "models"),
    results_csv_path=str(proj_dir / "python" / "results" / "training_results.csv"),
    history_plot_dir=str(proj_dir / "python" / "results" / "plots")
)

## Train model

In [ ]:
# Train for a specified number of epochs
trainer.train(epochs=15)

## Evaluate and Save Results

In [ ]:
trainer.evaluate()
trainer.save_results_to_csv()

Evaluation results - F1 (macro): 0.4617, F1 (weighted): 0.7919, AUC: 0.3859, Accuracy: 0.8576, Sensitivity: 0.0000, Specificity: 1.0000
